# Quadratic Bezier

A segment of $B^{(2)}$-Spline between $c_{j-1}, c_{j}, c_{j+1}$ is equivalent to $B^{(2)}$-ezier with $c_1 = ½(c_{j-1} + c_j), c_2 = c_j, c_3 = ⅓(c_{j+1} + c_j)$


In [ ]:
%%html
<style>
    :root {
        --jp-content-font-color0: var(--vscode-editor-foreground);
        --jp-content-font-color1: var(--vscode-editor-foreground);
        --jp-widgets-color: var(--vscode-editor-foreground);
        --jp-widgets-input-color: var(--vscode-editor-foreground);
        --jp-widgets-input-background-color: var(--vscode-editor-background);
        --jp-widgets-font-size: var(--vscode-editor-font-size);
    }
    .jupyter-widgets input {
        background-color: var(--jp-widgets-input-background-color);
    }
    .cell-output-ipywidget-background {
        background-color: transparent !important;
    }
</style>


In [ ]:
import numpy as np
from numpy.typing import NDArray
import ipywidgets as wg
import k3d

from utils import npvec, arr, arrgs, garr, f32, normalize

## Bezier

Centered around $c_2$

- $v_1 = c_1 - c_2$
- $v_3 = c_3 - c_2$
- $v_a = v_3 + v_1$ — parabolic axis
- $v_d = v_3 - v_1$ — parabolic direction = flow at 0.5

$$
S(t) - c_2 =
\big[1, t, t^2 \big]
\begin{pmatrix}
1 & 0 \\
-2 & 0 \\
1 &  1
\end{pmatrix}
\begin{bmatrix}
v_1 \\
v_3 \\
\end{bmatrix} =
\big[1, t, t^2 \big]
\begin{bmatrix}
v_1 \\
-2 v_1 \\
v_3 + v_1 \\
\tag{bezier form}
\end{bmatrix}
$$

$$
\frac{dS}{dt}(t) =
\big[1, t \big]
2
\begin{pmatrix}
-1 & 0 \\
1 & 1 \\
\end{pmatrix}
\begin{bmatrix}
v_1 \\
v_3 \\
\end{bmatrix}
=
2
\big[1, t \big]
\begin{bmatrix}
-v_1 \\
v_1 + v_3 \\
\end{bmatrix}
$$


In [ ]:
class Bezier:
    """Centered at c_2"""

    c1: npvec
    c2: npvec
    c3: npvec
    v1: npvec
    v3: npvec
    va: npvec
    vd: npvec

    def __init__(self, bcontrols: NDArray):
        p1 = bcontrols[0]
        p2 = bcontrols[1]
        p3 = bcontrols[2]
        self.c1 = (p1 + p2) * 0.5
        self.c2 = p2
        self.c3 = (p3 + p2) * 0.5
        self.v1 = self.c1 - self.c2
        self.v3 = self.c3 - self.c2
        self.va = self.v3 + self.v1
        self.vd = self.v3 - self.v1

    def localizep(self, point: npvec) -> npvec:
        return point - self.c2

    def globalizep(self, point: npvec) -> npvec:
        return self.c2 + point

    def point(self, t: float) -> npvec:
        return t * t * self.va - 2 * t * self.v1 + self.v1

    def flow(self, t: float) -> npvec:
        return 2 * (self.va * t - self.v1)

    def curve(self, tspace: NDArray) -> NDArray:
        return garr(self.point(t) for t in tspace)

## Time-Centered

Centered around $t = 0.5$

$$
S(t') - c_2 =
\big[1, t', t'^2 \big]
\begin{pmatrix}
.25 & .25 \\
-1 & 1 \\
1 & 1 \\
\end{pmatrix}
\begin{bmatrix}
v_1 \\
v_3 \\
\end{bmatrix}
=
\big[1, t', t'^2 \big]
\begin{bmatrix}
.25 v_a \\
v_d \\
v_a \\
\end{bmatrix}
$$

$$
\frac{dS}{dt}(t') =
\big[1, t' \big]
\begin{pmatrix}
-1 & 1 \\
2 & 2 \\
\end{pmatrix}
\begin{bmatrix}
v_1 \\
v_3 \\
\end{bmatrix}
=
\big[1, t' \big]
\begin{bmatrix}
v_d \\
2 v_a \\
\end{bmatrix}
$$


In [ ]:
class Beziet:
    """Centered at c_2 and t=0.5"""

    c1: npvec
    c2: npvec
    c3: npvec
    v1: npvec
    v3: npvec
    va: npvec
    vd: npvec

    def __init__(self, bcontrols: NDArray):
        p1 = bcontrols[0]
        p2 = bcontrols[1]
        p3 = bcontrols[2]
        self.c1 = (p1 + p2) * 0.5
        self.c2 = p2
        self.c3 = (p3 + p2) * 0.5
        self.v1 = self.c1 - self.c2
        self.v3 = self.c3 - self.c2
        self.va = self.v3 + self.v1
        self.vd = self.v3 - self.v1

    def localizep(self, point: npvec) -> npvec:
        return point - self.c2

    def globalizep(self, point: npvec) -> npvec:
        return self.c2 + point

    def point(self, t: float) -> npvec:
        return t * t * self.va + t * self.vd + 0.25 * self.va

    def flow(self, t: float) -> npvec:
        return 2 * self.va * t + self.vd

    def curve(self, tspace: NDArray) -> NDArray:
        return garr(self.point(t) for t in tspace)

## Local basis

Transforming to basis $v_d, v_a$:

- $v_d \to [1, 0]$
- $v_a \to [0, 1]$

$$
T =
\begin{pmatrix}
d_x & a_x \\
d_y & a_y \\
d_z & a_z \\
\end{pmatrix}
\tag{loc2wrl}
$$

In the local basis:

$$S'(t') - c_2 = [t', t'^2 + .25]$$
$$\frac{dS'}{dt}(t') = [1, 2t']$$


In [ ]:
class Beziel:
    """In basis of va, vd"""

    c1: npvec
    c2: npvec
    c3: npvec
    v1: npvec
    v3: npvec
    va: npvec
    vd: npvec

    T: NDArray

    def __init__(self, bcontrols: NDArray):
        p1 = bcontrols[0]
        p2 = bcontrols[1]
        p3 = bcontrols[2]
        self.c1 = (p1 + p2) * 0.5
        self.c2 = p2
        self.c3 = (p3 + p2) * 0.5
        self.v1 = self.c1 - self.c2
        self.v3 = self.c3 - self.c2
        self.va = self.v3 + self.v1
        self.vd = self.v3 - self.v1

        self.T = arr((self.vd, self.va)).T

    # def localizep(self, point: npvec) -> npvec:
    #     return point - self.c2

    def globalizep(self, point: npvec) -> npvec:
        return self.c2 + self.T @ point

    def globalizev(self, point: npvec) -> npvec:
        return self.T @ point

    def point(self, t: float) -> npvec:
        return arrgs(t, t * t + 0.25)

    def flow(self, t: float) -> npvec:
        return arrgs(1, 2 * t)

    def curve(self, tspace: NDArray) -> NDArray:
        return garr(self.point(t) for t in tspace)

---

# Plotting


In [ ]:
def random_controls():
    c2 = arrgs(0.0, 0.0, 1.0)
    return arrgs(
        c2 + (np.random.random(3) - 0.5) * arrgs(2.0, 2.0, 1.0),
        c2,
        c2 + (np.random.random(3) - 0.5) * arrgs(2.0, 2.0, 1.0),
    )

In [ ]:
# global
controls = arr(((-1, -1, 0.5), (0.0, 0.0, 1.0), (1, -1, 1.5)))

tspace = np.linspace(0.0, 1.0, 16, dtype=np.float32)
tspace0 = np.linspace(-0.5, +0.5, 16, dtype=np.float32)

In [ ]:
plot = k3d.Plot(
    height=720,
    background_color=0x404040,
    grid_color=0x383838,
    label_color=0x000000,
    menu_visibility=False,
    grid=(-1, -1, 0, 1, 1, 2),
    grid_auto_fit=False,
)
plot.layout = wg.Layout(width="720px", height="720px")

In [ ]:
toggle1 = wg.Checkbox(description="controls", value=True)
toggle2 = wg.Checkbox(description="tangents", value=False)

regenerate_btn = wg.Button(description="regenerate")
curvetype_sel = wg.RadioButtons(description="method", options=["Bezier", "Beziet", "Beziel"], value="Bezier")


In [ ]:
wg.HBox([plot, wg.VBox([toggle1, toggle2, regenerate_btn, curvetype_sel])], layout=dict(width="100%", grid_gap="8px"))

In [ ]:
k3curve = k3d.line(vertices=[], attribute=[], line_width=0.25, shader="mesh", color_map=k3d.colormaps.basic_color_maps.Rainbow, color_range=[0.0, 1.0])
k3controls = k3d.line(vertices=controls, color=0x808080, line_width=0.125, shader="thick", visible=True)
k3flow = k3d.vectors(origins=[], vectors=[], use_head=False, color=0x000000, head_color=0xF0F0F0, visible=False)

plot += k3curve
plot += k3controls
plot += k3flow

In [ ]:
def toggle(k, vis: bool):
    k.visible = vis


toggle1.observe(lambda ch: toggle(k3controls, ch.new), "value")
toggle2.observe(lambda ch: toggle(k3flow, ch.new), "value")

In [ ]:
def rerandomize():
    controls[:] = random_controls()
    k3controls.vertices = f32(controls)
    regenerate()


def regenerate_bezier():
    bezier = Bezier(controls)

    k3curve.vertices = [bezier.globalizep(p) for p in bezier.curve(tspace)]
    k3curve.attribute = tspace
    k3curve.color_range = (0.0, 1.0)
    k3flow.origins = garr(bezier.globalizep(bezier.point(t)) for t in (0.0, 0.5, 1.0))
    k3flow.vectors = garr(normalize(bezier.flow(t)) for t in (0.0, 0.5, 1.0)) * 0.5


def regenerate_beziet():
    bezier = Beziet(controls)

    k3curve.vertices = [bezier.globalizep(p) for p in bezier.curve(tspace0)]
    k3curve.attribute = tspace0
    k3curve.color_range = (-0.5, +0.5)
    k3flow.origins = garr(bezier.globalizep(bezier.point(t)) for t in (-0.5, 0.0, +0.5))
    k3flow.vectors = garr(normalize(bezier.flow(t)) for t in (-0.5, 0.0, +0.5)) * 0.5


def regenerate_beziel():
    bezier = Beziel(controls)

    k3curve.vertices = [bezier.globalizep(p) for p in bezier.curve(tspace0)]
    k3curve.attribute = tspace0
    k3curve.color_range = (-0.5, +0.5)
    k3flow.origins = garr(bezier.globalizep(bezier.point(t)) for t in (-0.5, 0.0, +0.5))
    k3flow.vectors = garr(normalize(bezier.globalizev(bezier.flow(t))) for t in (-0.5, 0.0, +0.5)) * 0.5


def regenerate():
    curvetype = curvetype_sel.value
    if curvetype == "Bezier":
        regenerate_bezier()
    elif curvetype == "Beziet":
        regenerate_beziet()
    elif curvetype == "Beziel":
        regenerate_beziel()


regenerate()

In [ ]:
regenerate_btn.on_click(lambda _: rerandomize())
curvetype_sel.observe(lambda _: regenerate(), "value")